# Cora → centroid hypergraph (Bai et al.)

Citation graph from `dissertation/cora/` (`cora.content`, `cora.cites`) is turned into a **hypergraph** following the construction used alongside **hypergraph convolution** in:

> Song Bai, Feihu Zhang, Philip H. S. Torr, *Hypergraph Convolution and Hypergraph Attention*, arXiv:1901.08150 (2019). https://arxiv.org/abs/1901.08150

**Construction (as in that line of work):**

1. Vertices = all papers (articles).
2. For each paper $v$ as **centroid**, one hyperedge contains $v$ and every paper that has a **citation link** with $v$ (either $v$ cites $u$ or $u$ cites $v$). Equivalently: hyperedge $e_v = \{v\} \cup N(v)$ in the **undirected** graph obtained by ignoring edge direction.

Node indices `0 … n-1` follow the **row order** in `cora.content` (same as bag-of-words features and labels).

The code cell writes a single file **`data/cora_centroid/hypergraph.pickle`**: a list of hyperedges as tuples of node indices (`edge_list[i]` = centroid `i`’s hyperedge), in the same order as rows of `cora.content`.

In [ ]:
# from pathlib import Path
# from collections import defaultdict
# import pickle
# from hypergraphx import Hypergraph

# cora_dir = Path("..") / "cora"
# if not (cora_dir / "cora.content").exists():
#     raise FileNotFoundError(cora_dir.resolve())

# paper_ids = []
# labels_raw = []
# with open(cora_dir / "cora.content", encoding="utf-8", errors="replace") as f:
#     for line in f:
#         parts = line.strip().split("\t")
#         paper_ids.append(parts[0])
#         labels_raw.append(parts[-1])

# n = len(paper_ids)
# pid_to_idx = {p: i for i, p in enumerate(paper_ids)}

# neigh = defaultdict(set)
# with open(cora_dir / "cora.cites", encoding="utf-8", errors="replace") as f:
#     for line in f:
#         cited, citing = line.strip().split()
#         if cited in pid_to_idx and citing in pid_to_idx:
#             u, v = pid_to_idx[cited], pid_to_idx[citing]
#             neigh[u].add(v)
#             neigh[v].add(u)

# edge_list = []
# for i in range(n):
#     nodes = {i} | neigh[i]
#     edge_list.append(tuple(sorted(nodes)))

# H = Hypergraph(edge_list)

# classes = sorted(set(labels_raw))
# cls_to_y = {c: j for j, c in enumerate(classes)}
# labels = [cls_to_y[c] for c in labels_raw]

# Cora → https://github.com/MingyuanLi88888/HP2PH/blob/main/HP2PH/

In [54]:
from pathlib import Path
import pickle

import numpy as np
from scipy.io import loadmat
from hypergraphx import Hypergraph

cora_mat_path = Path(
    "/Users/lusineadunts/Documents/omms/Hilary/dissertation/github_MingyuanLi88888_HP2PH/cora_cocitation.mat"
)
cora_cocitation = loadmat(cora_mat_path)
cora_mat = cora_cocitation["S"].toarray()

n_nodes, m_edges = cora_mat.shape
edge_list = []
for j in range(m_edges):
    nodes = np.flatnonzero(cora_mat[:, j]).astype(int).tolist()
    if nodes:
        edge_list.append(tuple(sorted(nodes)))

n_unique_sets = len(set(edge_list))
H_cora_incidence = Hypergraph(edge_list)
out_dir = Path("data") / "cora"
out_dir.mkdir(parents=True, exist_ok=True)
payload = {
    "edge_list": edge_list,
    "incidence_shape": cora_mat.shape,
    "source_mat": str(cora_mat_path),
    "n_hyperedges_paper": m_edges,
    "n_unique_hyperedge_sets": n_unique_sets,
}
out_path = out_dir / "cora.pickle"
with open(out_path, "wb") as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Incidence S: shape {cora_mat.shape} (n_nodes={n_nodes})")
print(f"Hyperedges = {m_edges} as in the paper (one per column of S).")
print(f"Unique vertex-sets among columns: {n_unique_sets}; hypergraphx |E|={len(H_cora_incidence.get_edges())} merges duplicates.")
print(f"Hypergraphx: |V|={len(H_cora_incidence.get_nodes())}")
print(f"Saved: {out_path.resolve()}")

Incidence S: shape (1330, 1503) (n_nodes=1330)
Hyperedges = 1503 as in the paper (one per column of S).
Unique vertex-sets among columns: 1413; hypergraphx |E|=1413 merges duplicates.
Hypergraphx: |V|=1330
Saved: /Users/lusineadunts/Documents/omms/Hilary/dissertation/hypergraphs/data/cora/cora.pickle


In [ ]:
from pathlib import Path

from scipy.io import loadmat

contact_mat_path = Path(
    "/Users/lusineadunts/Documents/omms/Hilary/dissertation/github_MingyuanLi88888_HP2PH_1/contact_highschool.mat"
)
if not contact_mat_path.is_file():
    raise FileNotFoundError(contact_mat_path)

contact_raw = loadmat(contact_mat_path)
contact_data = {k: v for k, v in contact_raw.items() if not k.startswith("__")}

print("Loaded:", contact_mat_path)
print("Keys:", list(contact_data.keys()))
for k, v in contact_data.items():
    print(f"  {k}: {type(v).__name__}", getattr(v, "shape", ""))